**Support Ticket Category Classifier using NLP**

1. Support Ticket Category Classifier using NLP and Machine Learning

## Objective
Classify customer support tickets into categories using Natural Language Processing.

Workflow:
1. Load dataset
2. Clean text
3. Prepare data
4. TF-IDF feature extraction
5. Train ML model
6. Evaluate model
7. Save model

2.# import libraries

In [3]:
import pandas as pd
import numpy as np
import re
import pickle

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt

3.upload dataset

In [4]:
from google.colab import files

uploaded = files.upload()

Saving customer_support_tickets.csv to customer_support_tickets.csv


4.read dataset

In [5]:
filename = list(uploaded.keys())[0]

df = pd.read_csv(filename)

df.head()

,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


5.dataset info

In [6]:
print(df.shape)

df.info()

(8469, 17)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8469 entries, 0 to 8468
Data columns (total 17 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Ticket ID                     8469 non-null   int64  
 1   Customer Name                 8469 non-null   object 
 2   Customer Email                8469 non-null   object 
 3   Customer Age                  8469 non-null   int64  
 4   Customer Gender               8469 non-null   object 
 5   Product Purchased             8469 non-null   object 
 6   Date of Purchase              8469 non-null   object 
 7   Ticket Type                   8469 non-null   object 
 8   Ticket Subject                8469 non-null   object 
 9   Ticket Description            8469 non-null   object 
 10  Ticket Status                 8469 non-null   object 
 11  Resolution                    2769 non-null   object 
 12  Ticket Priority               8469 non-null   objec

6.check missing values

In [7]:
df.isnull().sum()

,0
Ticket ID,0
Customer Name,0
Customer Email,0
Customer Age,0
Customer Gender,0
Product Purchased,0
Date of Purchase,0
Ticket Type,0
Ticket Subject,0
Ticket Description,0


7. select required columns

In [23]:
def create_category(row):

    text = (
        str(row['Ticket Subject']) + " " +
        str(row['Ticket Description'])
    ).lower()


    if "refund" in text or "money back" in text:
        return "Refund request"

    elif "cancel" in text or "cancellation" in text:
        return "Cancellation request"

    elif "billing" in text or "payment" in text or "charge" in text:
        return "Billing inquiry"

    elif "setup" in text or "install" in text or "compatibility" in text:
        return "Technical issue"

    else:
        return "Product inquiry"

In [24]:
df['new_category'] = df.apply(
    create_category,
    axis=1
)

8.category distribution

In [25]:
df['new_category'].value_counts()

,count
new_category,
Product inquiry,4343
Technical issue,2051
Billing inquiry,844
Refund request,727
Cancellation request,504


9.train the model

In [29]:
from sklearn.utils import resample

# Find the smallest class size
min_count = data['category'].value_counts().min()

balanced_data = pd.DataFrame()

for category in data['category'].unique():

    category_data = data[data['category'] == category]

    category_sample = resample(
        category_data,
        replace=False,
        n_samples=min_count,
        random_state=42
    )

    balanced_data = pd.concat(
        [balanced_data, category_sample]
    )


data = balanced_data.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)


data['category'].value_counts()

,count
category,
Billing inquiry,504
Technical issue,504
Product inquiry,504
Cancellation request,504
Refund request,504


10.text cleaning function

In [26]:
data = df.copy()

data['text'] = (
    data['Ticket Subject'].astype(str)
    + " "
    + data['Ticket Description'].astype(str)
    + " "
    + data['Product Purchased'].astype(str)
)

data['category'] = data['new_category']

data = data[['text','category']]

data.head()

,text,category
0,Product setup I'm having an issue with the {pr...,Billing inquiry
1,Peripheral compatibility I'm having an issue w...,Technical issue
2,Network problem I'm facing a problem with my {...,Billing inquiry
3,Account access I'm having an issue with the {p...,Product inquiry
4,Data loss I'm having an issue with the {produc...,Product inquiry


11.apply cleaning

In [11]:
data['clean_text'] = data['text'].apply(clean_text)

data.head()

,text,category,clean_text
0,I'm having an issue with the {product_purchase...,Technical issue,i m having an issue with the product purchased...
1,I'm having an issue with the {product_purchase...,Technical issue,i m having an issue with the product purchased...
2,I'm facing a problem with my {product_purchase...,Technical issue,i m facing a problem with my product purchased...
3,I'm having an issue with the {product_purchase...,Billing inquiry,i m having an issue with the product purchased...
4,I'm having an issue with the {product_purchase...,Billing inquiry,i m having an issue with the product purchased...


13.split data

In [31]:
X = data['clean_text']

y = data['category']


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

14. TF-IDF verification

In [32]:
tfidf = TfidfVectorizer(

    max_features=20000,

    ngram_range=(1,2),

    stop_words='english'

)


X_train_tfidf = tfidf.fit_transform(
    X_train
)


X_test_tfidf = tfidf.transform(
    X_test
)


print(
    X_train_tfidf.shape
)

(2016, 16358)


15. train model

In [48]:
from sklearn.linear_model import LogisticRegression


model = LogisticRegression(
    max_iter=1000
)


model.fit(
    X_train_tfidf,
    y_train
)

LogisticRegression(max_iter=1000)

16. evaluation

In [59]:
from sklearn.metrics import accuracy_score, f1_score, classification_report


y_pred = model.predict(
    X_test_tfidf
)


print(
    "Accuracy:",
    accuracy_score(y_test, y_pred)
)


print(
    "F1 Score:",
    f1_score(
        y_test,
        y_pred,
        average="weighted"
    )
)


print(
    classification_report(
        y_test,
        y_pred
    )
)

Accuracy: 0.9186507936507936
F1 Score: 0.9198742014986095
                      precision    recall  f1-score   support

     Billing inquiry       0.99      0.90      0.94       101
Cancellation request       0.98      0.93      0.95       100
     Product inquiry       0.80      1.00      0.89       101
      Refund request       1.00      0.84      0.91       101
     Technical issue       0.89      0.92      0.90       101

            accuracy                           0.92       504
           macro avg       0.93      0.92      0.92       504
        weighted avg       0.93      0.92      0.92       504



17. confusion matrix

In [53]:
cm = confusion_matrix(
    y_test,
    y_pred
)

cm

array([[ 91,   0,   7,   0,   3],
       [  0,  93,   5,   0,   2],
       [  0,   0, 101,   0,   0],
       [  1,   2,   6,  85,   7],
       [  0,   0,   8,   0,  93]])

18.test prediction

In [54]:
sample = [
    "My payment was deducted twice and I need help with my transaction"
]


sample_clean = [
    clean_text(sample[0])
]


sample_vector = tfidf.transform(
    sample_clean
)


prediction = model.predict(
    sample_vector
)


print(
    prediction[0]
)

Billing inquiry


19. model saving

In [60]:
import pickle


pickle.dump(
    model,
    open("ticket_classifier.pkl","wb")
)


pickle.dump(
    tfidf,
    open("tfidf_vectorizer.pkl","wb")
)

In [61]:
print(classification_report(y_test, y_pred))

                      precision    recall  f1-score   support

     Billing inquiry       0.99      0.90      0.94       101
Cancellation request       0.98      0.93      0.95       100
     Product inquiry       0.80      1.00      0.89       101
      Refund request       1.00      0.84      0.91       101
     Technical issue       0.89      0.92      0.90       101

            accuracy                           0.92       504
           macro avg       0.93      0.92      0.92       504
        weighted avg       0.93      0.92      0.92       504



In [57]:
import os

print(os.listdir())

['.config', 'tfidf_vectorizer.pkl', 'ticket_classifier.pkl', 'customer_support_tickets.csv', 'sample_data']


downloading files

In [62]:
from google.colab import files

files.download("ticket_classifier.pkl")
files.download("tfidf_vectorizer.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#**Conclusion**


The Support Ticket Category Classifier successfully classifies customer support tickets using NLP and Machine Learning.

The final Logistic Regression model with TF-IDF features achieved approximately 92% accuracy and F1 score.

The trained model is integrated with a Streamlit application for real-time ticket category and urgency prediction.

Model Performance:

Accuracy: 92%

F1 Score: 92%

Model:
Logistic Regression with TF-IDF features